In [0]:
from pyspark.sql import functions as F

# Read the Bronze Delta table
bronze_df = spark.table("workspace.default.bronze_wind_power")

# Display the schema
bronze_df.printSchema()

# Display a sample of the data
display(bronze_df.limit(10))

In [0]:
silver_df = bronze_df.withColumn(
    "measurement_timestamp",
    F.to_timestamp(
        F.concat_ws(" ", F.col("date"), F.col("time")),
        "yyyy-MM-dd HH-mm-ss"
    )
)

In [0]:
display(silver_df.limit(10))

In [0]:
# Count null values in every column
null_counts = silver_df.select([
    F.count(
        F.when(F.col(column).isNull(), 1)
    ).alias(column)
    for column in silver_df.columns
])
display(null_counts)
print(len(silver_df.columns))
print(silver_df.count())

In [0]:
# Define the target Silver table
silver_table = "workspace.default.silver_wind_power"

# Save the transformed DataFrame as a Delta table
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

print(f"Silver table created successfully: {silver_table}")

In [0]:
# Read the persisted Silver table
silver_check_df = spark.table(
    "workspace.default.silver_wind_power"
)

# Validate row count and column count
print("Number of rows:", silver_check_df.count())
print("Number of columns:", len(silver_check_df.columns))

# Inspect the schema and sample records
silver_check_df.printSchema()

display(silver_check_df.limit(10))